# Phase 3 — three datasets, two architectures, three seeds

<a href="https://colab.research.google.com/github/fatemehsalmani75/active-subspace-bloodmnist/blob/main/notebooks/phase3_multi_dataset.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Phases 1 and 2 were run on BloodMNIST with VGG-19 only. This notebook checks whether the findings hold on
**BloodMNIST, DermaMNIST and PathMNIST**, with **VGG-19 and ResNet-18**.

**Part A — attack strength** (every dataset × architecture × seed 0, 1, 2, so 18 models): the AS universal
attack against two baselines, UAP (DeepFool, ART) and SGD-UAP (Shafahi et al., 2020).

**Part B — defenses and the gradient-size predictor** (every dataset × architecture, seed 0): five fine-tuned
models per base model (AS fine-tuning, UAT, PGD-AT, trace penalty β = 0.1 and β = 1), so 36 models with the bases.

**Rules, written before the run:**
- **R1, attack.** The AS attack is the stronger universal attack if its mean fooling rate is above both baselines
  in at least 5 of the 6 dataset–architecture settings, and a paired Wilcoxon test over the 18 Part A models
  gives p < 0.05 against each baseline.
- **R2, predictor.** $\operatorname{tr} C$ predicts universal vulnerability if the Spearman ρ between
  $\operatorname{tr} C$ and AS fooling, computed inside each setting over its 6 models, has a median ≥ 0.7 over
  the 6 settings. The same number is reported for $\lambda_1$ for comparison.
- **R3, defense.** The trace penalty is a competitive cheap defense if, in at least 4 of the 6 settings, one of its
  two models has AS fooling within 5 points of UAT while losing at most 2 points of clean accuracy against the base.

**How to run.** First run only the second cell (the one with `drive.mount`) and press **Allow**; then *Run all*
with a T4 GPU. The whole notebook takes about 10 hours of GPU time, so it will need several Colab sessions:
every model and every result row is saved in Google Drive (`active_subspace_runs/phase3`) as soon as it is
ready, and *Run all* again continues where it stopped. To finish one dataset at a time, edit `DATASETS` in the
settings cell.

In [ ]:
%cd /content
!test -d active-subspace-bloodmnist || git clone -q https://github.com/fatemehsalmani75/active-subspace-bloodmnist.git
%cd /content/active-subspace-bloodmnist
!git pull -q
!pip install -q medmnist adversarial-robustness-toolbox

In [ ]:
import os, copy, sys, time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, wilcoxon

from asnet.data import CLASSES, get_datasets, make_loader, subset
from asnet.models import build_model
from asnet.utils import set_seed, get_device, train_classifier, stack_dataset
from asnet.attack import as_universal_attack, fooling_rate, predict
from asnet.uap_baseline import uap_baseline, sgd_uap
from asnet.robust import adv_finetune, uat_finetune, pgd_finetune, spectral_finetune, as_perturbation
from asnet.spectrum import input_spectrum

OUT = "/content/drive/MyDrive/active_subspace_runs/phase3"
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.isdir("/content/drive/MyDrive"):
        raise RuntimeError("Google Drive is NOT connected. Run this cell again and press Allow.")
else:
    OUT = "runs/phase3"
MODELS = os.path.join(OUT, "models")
os.makedirs(MODELS, exist_ok=True)
CSV_A = os.path.join(OUT, "phase3_attacks.csv")
CSV_B = os.path.join(OUT, "phase3_defenses.csv")
print("Saving everything to", OUT)

device = get_device()
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(device)

## Settings (the same as Phases 1 and 2 where they overlap)

In [ ]:
DATASETS = ["bloodmnist", "dermamnist", "pathmnist"]
ARCHS = ["vgg19", "resnet18"]
SEEDS = [0, 1, 2]
SIZE, EPOCHS, FT_EPOCHS = 64, 20, 5
MAX_TRAIN = 12000          # PathMNIST has 90k training images; a fixed random 12k keeps the cost close to BloodMNIST
DELTA, REFRESH, PGD_EPS = 4, 60, 2.0
N_ATTACK, N_SPECTRUM = 200, 2000
DEFENSES = ["AS fine-tuning", "UAT", "PGD-AT", "trace beta=0.1", "trace beta=1"]

## Helpers

In [ ]:
_data = {}
def data(name):
    if name not in _data:
        train_aug, _, _ = get_datasets(SIZE, augment=True, name=name)
        train, val, test = get_datasets(SIZE, name=name)
        if len(train) > MAX_TRAIN:
            train_aug, train = subset(train_aug, MAX_TRAIN, seed=7), subset(train, MAX_TRAIN, seed=7)
        Xte, yte = stack_dataset(test, device=device)
        _data[name] = dict(train=train, val=val, Xte=Xte, yte=yte,
                           loader=make_loader(train_aug, 128, shuffle=True),
                           spec=make_loader(subset(train, N_SPECTRUM, seed=10), 100))
    return _data[name]

def ckpt(*parts):
    return os.path.join(MODELS, "_".join(str(p) for p in parts).replace(" ", "").replace("=", "") + ".pt")

def load_or_train(path, arch, nc, train_fn):
    m = build_model(arch, nc, SIZE)
    if os.path.exists(path):
        ck = torch.load(path, map_location="cpu"); m.load_state_dict(ck["state_dict"]); t = ck["time"]
    else:
        t0 = time.time(); m = train_fn(m); t = time.time() - t0
        torch.save({"state_dict": m.state_dict(), "time": t}, path)
    return m.to(device).eval(), t

def evaluate_model(m, d, nc, seed):
    for p in m.parameters():
        p.requires_grad_(False)
    pred = predict(m, d["Xte"])
    per_class = [(pred[d["yte"] == c] == c).float().mean().item() for c in range(nc) if (d["yte"] == c).any()]
    Xs, ys = stack_dataset(subset(d["train"], N_ATTACK, seed=500 + seed), device=device)
    s = input_spectrum(m, d["spec"], device=device)
    row = {"clean acc": (pred == d["yte"]).float().mean().item(), "balanced acc": float(np.mean(per_class)),
           "lambda1": s["lambda1"], "trace": s["trace"], "lambda1_share": s["lambda1_share"],
           "AS fooling": fooling_rate(m, d["Xte"], as_universal_attack(m, Xs, ys, delta=DELTA, seed=seed, log=lambda *a: None).v),
           "UAP fooling": fooling_rate(m, d["Xte"], uap_baseline(m, Xs, nc, delta=DELTA, device=device)),
           "SGD-UAP fooling": fooling_rate(m, d["Xte"], sgd_uap(m, Xs, ys, delta=DELTA, seed=seed))}
    for p in m.parameters():
        p.requires_grad_(True)
    return row

def append(csv, rows, row):
    rows.append(row); pd.DataFrame(rows).to_csv(csv, index=False)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})

## Part A — base models and attack strength (18 models)

In [ ]:
rows_a = pd.read_csv(CSV_A).to_dict("records") if os.path.exists(CSV_A) else []
done_a = {(r["dataset"], r["arch"], r["seed"]) for r in rows_a}

for ds in DATASETS:
    d, nc = data(ds), CLASSES[ds]
    for arch in ARCHS:
        for seed in SEEDS:
            if (ds, arch, seed) in done_a:
                print(ds, arch, seed, "already evaluated"); continue
            print(f"--- {ds} {arch} seed {seed}")
            path = ckpt(ds, arch, "base", seed)
            def train_fn(m, seed=seed):
                set_seed(seed)
                return train_classifier(m, d["loader"], make_loader(d["val"], 256), epochs=EPOCHS, device=device,
                                        log=lambda *a: None)[0]
            m, t = load_or_train(path, arch, nc, train_fn)
            append(CSV_A, rows_a, {"dataset": ds, "arch": arch, "seed": seed, "train time (s)": t,
                                   **evaluate_model(m, d, nc, seed)})
            del m; torch.cuda.empty_cache()

res_a = pd.DataFrame(rows_a)
res_a.groupby(["dataset", "arch"])[["clean acc", "balanced acc", "AS fooling", "UAP fooling", "SGD-UAP fooling"]].agg(["mean", "std"]).round(3)

## Part B — defended models (seed 0, 30 models)

In [ ]:
rows_b = pd.read_csv(CSV_B).to_dict("records") if os.path.exists(CSV_B) else []
done_b = {(r["dataset"], r["arch"], r["model"]) for r in rows_b}

for ds in DATASETS:
    d, nc = data(ds), CLASSES[ds]
    for arch in ARCHS:
        base_path = ckpt(ds, arch, "base", 0)
        if not os.path.exists(base_path):
            print(ds, arch, "no base model yet: run Part A first"); continue
        base, _ = load_or_train(base_path, arch, nc, None)
        if (ds, arch, "none (base)") not in done_b:
            a = res_a[(res_a.dataset == ds) & (res_a.arch == arch) & (res_a.seed == 0)].iloc[0].to_dict()
            append(CSV_B, rows_b, {**a, "model": "none (base)", "train time (s)": 0.0})
        Xa, ya = stack_dataset(subset(d["train"], 100, seed=300), device=device)
        for name in DEFENSES:
            if (ds, arch, name) in done_b:
                print(ds, arch, name, "already evaluated"); continue
            print(f"--- {ds} {arch} {name}")
            path = ckpt(ds, arch, name, 0)
            def train_fn(_, name=name):
                set_seed(1); m = copy.deepcopy(base)
                if name == "AS fine-tuning":
                    return adv_finetune(m, d["loader"], as_perturbation(Xa, ya, DELTA), epochs=FT_EPOCHS, refresh=REFRESH, device=device, log=lambda *a: None)[0]
                if name == "UAT":
                    return uat_finetune(m, d["loader"], DELTA, epochs=FT_EPOCHS, device=device, log=lambda *a: None)[0]
                if name == "PGD-AT":
                    return pgd_finetune(m, d["loader"], PGD_EPS, epochs=FT_EPOCHS, device=device, log=lambda *a: None)[0]
                beta = float(name.split("=")[1])
                return spectral_finetune(m, d["loader"], beta, mode="trace", epochs=FT_EPOCHS, device=device, log=lambda *a: None)[0]
            m, t = load_or_train(path, arch, nc, train_fn)
            append(CSV_B, rows_b, {"dataset": ds, "arch": arch, "seed": 0, "model": name, "train time (s)": t,
                                   **evaluate_model(m, d, nc, 0)})
            del m; torch.cuda.empty_cache()
        del base; torch.cuda.empty_cache()

res_b = pd.DataFrame(rows_b)
res_b.pivot_table(index=["dataset", "arch"], columns="model", values="AS fooling").round(3)

## The decision

In [ ]:
lines = []
settings = [(ds, a) for ds in DATASETS for a in ARCHS]

# R1
g = res_a.groupby(["dataset", "arch"])[["AS fooling", "UAP fooling", "SGD-UAP fooling"]].mean()
wins = sum((r["AS fooling"] > r["UAP fooling"]) and (r["AS fooling"] > r["SGD-UAP fooling"]) for _, r in g.iterrows())
p_uap = wilcoxon(res_a["AS fooling"], res_a["UAP fooling"], alternative="greater").pvalue
p_sgd = wilcoxon(res_a["AS fooling"], res_a["SGD-UAP fooling"], alternative="greater").pvalue
ok1 = wins >= 5 and p_uap < 0.05 and p_sgd < 0.05
lines.append(f"R1 attack: AS beats both baselines in {wins}/{len(g)} settings; Wilcoxon p = {p_uap:.4f} (vs UAP), "
             f"{p_sgd:.4f} (vs SGD-UAP) over {len(res_a)} models -> {'SUPPORTED' if ok1 else 'NOT SUPPORTED'}")

# R2
rho = []
for (ds, a), grp in res_b.groupby(["dataset", "arch"]):
    if len(grp) >= 4:
        rho.append({"dataset": ds, "arch": a, "n": len(grp),
                    "rho trace": spearmanr(grp["trace"], grp["AS fooling"])[0],
                    "rho lambda1": spearmanr(grp["lambda1"], grp["AS fooling"])[0]})
rho = pd.DataFrame(rho)
display(rho.round(3))
if len(rho):
    mt, ml = rho["rho trace"].median(), rho["rho lambda1"].median()
    lines.append(f"R2 predictor: median within-setting Spearman rho, trace = {mt:.3f}, lambda1 = {ml:.3f} over {len(rho)} settings "
                 f"-> trace {'SUPPORTED' if mt >= 0.7 and len(rho) == 6 else 'NOT SUPPORTED' if len(rho) == 6 else 'INCOMPLETE'}")
    rho.to_csv(os.path.join(OUT, "phase3_correlations.csv"), index=False)

# R3
good, n3 = 0, 0
for (ds, a), grp in res_b.groupby(["dataset", "arch"]):
    grp = grp.set_index("model")
    if not {"none (base)", "UAT", "trace beta=0.1", "trace beta=1"} <= set(grp.index):
        continue
    n3 += 1
    base_acc, uat = grp.loc["none (base)", "clean acc"], grp.loc["UAT", "AS fooling"]
    good += any(grp.loc[t, "AS fooling"] <= uat + 0.05 and grp.loc[t, "clean acc"] >= base_acc - 0.02
                for t in ["trace beta=0.1", "trace beta=1"])
lines.append(f"R3 defense: trace penalty competitive with UAT in {good}/{n3} settings "
             f"-> {'SUPPORTED' if good >= 4 and n3 == 6 else 'NOT SUPPORTED' if n3 == 6 else 'INCOMPLETE'}")

print("\n".join(lines))
open(os.path.join(OUT, "phase3_decision.txt"), "w").write("\n".join(lines))

## Figures

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.2))
labels, x = [], np.arange(len(g))
for i, (atk, c) in enumerate([("AS fooling", "#3b6ea5"), ("UAP fooling", "#d9822b"), ("SGD-UAP fooling", "#4a9b6e")]):
    s = res_a.groupby(["dataset", "arch"])[atk]
    ax.bar(x + (i - 1) * 0.27, s.mean() * 100, 0.27, yerr=s.std() * 100, capsize=3, color=c, label=atk.replace(" fooling", ""))
ax.set_xticks(x, [f"{ds.replace('mnist', '')}\n{a}" for ds, a in g.index])
ax.set(ylabel=f"universal fooling (%), delta = {DELTA}", title="universal attacks, mean ± std over 3 seeds")
ax.legend(); plt.tight_layout(); plt.savefig(os.path.join(OUT, "phase3_attacks.png"), bbox_inches="tight"); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
colors = plt.cm.tab10(np.arange(6))
for ax, meas, lab in [(axes[0], "trace", r"$\mathrm{tr}\, C$"), (axes[1], "lambda1", r"$\lambda_1$")]:
    for c, ((ds, a), grp) in zip(colors, res_b.groupby(["dataset", "arch"])):
        ax.scatter(grp[meas], grp["AS fooling"] * 100, color=c, s=45, label=f"{ds.replace('mnist', '')} {a}")
    ax.set_xscale("log"); ax.set(xlabel=lab, title=f"{lab} vs AS fooling, 36 models")
axes[0].set_ylabel("AS universal fooling (%)"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "phase3_predictor.png"), bbox_inches="tight"); plt.show()